### Project Overview
This notebook implements a full end-to-end pipeline for the CSIRO Image2Biomass prediction task. 
https://www.kaggle.com/competitions/csiro-biomass/overview

The goal is to predict five biomass-related targets from plot images and associated metadata, with a competition metric based on Kaggle-style weighted R² over all targets.

raw CSV + JPEGs → processed folds → baselines → hybrid teacher → distilled student


In [1]:
# ==================== 0. Setup Reproducibility & Config ====================
import os, json, random, platform
from dataclasses import dataclass, asdict
from pathlib import Path

# Make TensorFlow as deterministic as possible (given hardware / ops)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1" 
os.environ["PYTHONHASHSEED"] = str(42) 

import tensorflow as tf

tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
import cv2 # OpenCV (Open Source Computer Vision Library)
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedGroupKFold
from datetime import datetime

# Log library versions for reproducibility
versions = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "keras": tf.keras.__version__,
}
print("versions:", versions)

@dataclass
class CFG:
    """Global configuration for the CSIRO Image2Biomass project.

    This dataclass centralizes all tunable knobs and file paths so that
    the rest of the notebook can read configuration from a single object.

    Attributes:
        project_title: Human-readable project title.
        seed: Global random seed used for NumPy, Python, and TensorFlow.
        image_size_h: Height of input images in pixels.
        image_size_w: Width of input images in pixels.
        n_folds: Number of cross-validation folds.
        batch_size: Batch size for training and evaluation.
        raw_csv: Relative path to the original long-format training CSV.
        image_root: Root directory for image files referenced in the CSV.
        artifacts_dir: Directory where processed CSVs, configs, and models
            are saved.
        figs_dir: Directory where figures and diagnostic plots are saved.
        use_log1p: Whether to use log1p-transformed labels during training.
    """
    project_title: str = "CSIRO Image2Biomass Prediction"
    seed: int = 42
    image_size_h: int = 224
    image_size_w: int = 448 
    n_folds: int = 5
    batch_size: int = 8 # batch_size=8 for 357 images can be stable and better generalization
    raw_csv: str = "csiro-biomass/train.csv"
    image_root: str = "csiro-biomass"
    artifacts_dir: str = "artifacts/"
    figs_dir: str = "figs"
    use_log1p: bool = False

CFG = CFG()
Path(CFG.artifacts_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.figs_dir).mkdir(parents=True, exist_ok=True)

# ---------- Targets ----------
CORE_TARGETS = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g"]  # main modeling targets
TARGETS  = CORE_TARGETS + ["GDM_g","Dry_Total_g"]       # extra targets for EDA

# ---------- Reproducibility ----------
def set_seed(seed=42):
    """Set global random seed for Python, NumPy, and TensorFlow"""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(CFG.seed)

versions: {'python': '3.12.6', 'tensorflow': '2.17.0', 'numpy': '1.26.4', 'pandas': '2.3.1', 'sklearn': '1.7.1', 'keras': '3.12.0'}


In [2]:
# ==================== 1. read raw data ====================
df_raw = pd.read_csv(CFG.raw_csv)
display(df_raw.head(5))
# add tensor columns: abs_image_path & image_id
df_raw["abs_image_path"] = df_raw["image_path"].apply(lambda p: str(Path(CFG.image_root) / p))
df_raw["image_id"] = df_raw["abs_image_path"].map(lambda p: Path(p).stem)


FileNotFoundError: [Errno 2] No such file or directory: 'csiro-biomass/train.csv'

In [ ]:
# ==================== 2. Pivot ====================
def build_pivot_df(df):
    """convert long-format biomass table to a wide per-image table."""
    
    # 1) create wide table which has one row per image with all 5 targets as columns
    wide = (
        df.pivot_table(index="image_id", # → each img is a row
                       columns="target_name", # → the target columns to create 
                       values="target",  # → the values to fill the table
                       aggfunc="first") # → in case of duplicates, take the first
          .reset_index() # → add the index column, so image_id becomes a column again
    )
    
    # 2)target columns check ensure all present
    for col in TARGETS:
        if col not in wide.columns:
            wide[col] = np.nan # add missing value target columns with NaN values
    wide = wide[["image_id"] + TARGETS]

    # Join metadata: from original df extract unique rows
    meta_cols = ["image_id", "abs_image_path", "Sampling_Date", "State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm"]
    meta = df[meta_cols].drop_duplicates("image_id") # drop duplicate rows based on image_id
    out = wide.merge(meta, on="image_id", how="left") # merge: use wide as base, left join meta

    cols =  meta_cols + TARGETS # reorder columns: metadata → targets(features first, then targets)
    return out[cols]

# create and save (CSV)
df_pivot = build_pivot_df(df_raw)

out_dir = Path(CFG.artifacts_dir)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "train_wide.csv"
df_pivot.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Pivot shape:", df_pivot.shape)


Saved: artifacts/train_wide.csv
Pivot shape: (357, 12)


In [ ]:

# ==================== 3. Date features (update df_pivot)====================
from datetime import datetime
# 1) Date features
df_pivot["Sampling_Date"] = pd.to_datetime(df_pivot["Sampling_Date"], format="%Y/%m/%d", errors="coerce")
df_pivot["year"] = df_pivot["Sampling_Date"].dt.year
df_pivot["month"] = df_pivot["Sampling_Date"].dt.month
df_pivot["day_of_year"] = df_pivot["Sampling_Date"].dt.dayofyear

# 2) cyclical encodings
df_pivot['month_sin'] = np.sin(2 * np.pi * df_pivot['month'] / 12)
df_pivot['month_cos'] = np.cos(2 * np.pi * df_pivot['month'] / 12)
df_pivot['day_of_year_sin'] = np.sin(2 * np.pi * df_pivot['day_of_year'] / 365.25)
df_pivot['day_of_year_cos'] = np.cos(2 * np.pi * df_pivot['day_of_year'] / 365.25)

# 3) Categorical encoding (for tabular model)
# Convert "State" and "Species" features into numbers and save the mapping relationship for use during inference!
state_cat = df_pivot["State"].astype("category")    # 1. convert to category dtype
species_cat = df_pivot["Species"].astype("category")
df_pivot["State_encoded"] = state_cat.cat.codes     # 2. extract integer code as new column
df_pivot["Species_encoded"] = species_cat.cat.codes
STATE_MAP = dict(enumerate(state_cat.cat.categories)) # 3. create mapping dicts → int to category
SPECIES_MAP = dict(enumerate(species_cat.cat.categories))

# Reorder columns: metadata → targets
feature_cols = [col for col in df_pivot.columns if col not in TARGETS]
df_pivot = df_pivot[feature_cols + TARGETS]  # reorder columns: features first, then targets


In [ ]:
# ==================== 4. StratifiedKFold by Species + Dry_Total_g ====================

from sklearn.model_selection import StratifiedKFold

# 1) Build stratification labels: Species + binned Dry_Total_g
y_for_bins = df_pivot["Dry_Total_g"]
if y_for_bins.isna().any():
    y_for_bins = (
        df_pivot["Dry_Green_g"] 
        + df_pivot["Dry_Dead_g"] 
        + df_pivot["Dry_Clover_g"]
    )

# Bin Dry_Total_g into 5 quantile-based buckets
bins = pd.qcut(y_for_bins, q=5, labels=False, duplicates="drop")

# Stratification label = species name + biomass bin index
strata = df_pivot["Species"].astype(str) + "_" + bins.astype(str)

# 2) StratifiedKFold: split into 5 folds, shuffled, stratified by `strata`
#    This is what makes each fold about 357 / 5 ≈ 71 samples.
skf = StratifiedKFold(
    n_splits=CFG.n_folds,   # decide how many folds → ~71 samples per fold
    shuffle=True,
    random_state=546195, # 7499
)

folds = np.full(len(df_pivot), -1)

for fold, (_, val_idx) in enumerate(skf.split(df_pivot, y=strata)):
    folds[val_idx] = fold

df_pivot["fold"] = folds.astype(int)

# 3) Sanity check: no species should appear in only a single fold
violations = []
species_counts = df_pivot["Species"].value_counts()

for sp, total in species_counts.items():
    per_fold = df_pivot[df_pivot["Species"] == sp]["fold"].value_counts()
    for fold_id, cnt in per_fold.items():
        if cnt == total:  # if this species is entirely contained in one fold
            violations.append((sp, fold_id, int(cnt), int(total)))

if violations:
    print("[Warn] Some species only appear in a single fold:")
    for sp, f, c, tot in violations:
        print(f"  - Species={sp}, fold={f}, count={c}/{tot}")
    # If you want to enforce this as a hard constraint, you can replace the
    # prints above with: `assert not violations`

# 4) Report: distribution of Dry_Total_g in each fold (original scale)
def fold_report(df):
    tgts = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]
    return df.groupby("fold")[tgts].agg(["mean","std","min","max","count"]).round(2)

print("\n=== Fold report (on original scale, before any log transform) ===")
display(fold_report(df_pivot))

print("\n=== Fold sizes ===")
for fold_num in range(CFG.n_folds):
    print(f"Fold {fold_num}: {(df_pivot['fold'] == fold_num).sum()} samples")

# 5) save
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)
print("Saved processed with folds to:", processed_path)


=== Fold report (on original scale, before any log transform) ===


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Dry_Green_g                            Dry_Dead_g                     \
            mean    std   min     max count       mean    std  min    max   
fold                                                                        
0          25.67  22.78  0.00  129.34    72      13.57  15.35  0.0  83.84   
1          25.85  24.08  0.00  116.75    72      11.62  12.07  0.0  50.23   
2          28.72  29.14  0.00  119.60    71      10.11  11.21  0.0  53.26   
3          24.88  22.17  0.31   83.93    71      13.34  11.78  0.0  46.07   
4          28.04  28.56  0.00  157.98    71      11.57  11.09  0.0  52.93   

            ...  GDM_g                            Dry_Total_g               \
     count  ...   mean    std   min     max count        mean    std   min   
fold        ...                                                              
0       72  ...  31.24  21.65  1.40  129.34    72       44.80  28.16  6.30   
1       72  ...  33.00  24.31  3.78  116.75    72       44.63  24.36  9.20   
2       71  ...  36.44  27.72  1.80  119.60    71       46.54  31.75  2.48   
3       71  ...  32.14  22.23  2.40   96.76    71       45.49  24.89  4.30   
4       71  ...  33.59  28.45  1.04  157.98    71       45.15  30.77  1.04   

                   
        max count  
fold               
0     157.9    72  
1     120.4    72  
2     166.1    71  
3     119.1    71  
4     185.7    71  

[5 rows x 25 columns]


=== Fold sizes ===
Fold 0: 72 samples
Fold 1: 72 samples
Fold 2: 71 samples
Fold 3: 71 samples
Fold 4: 71 samples
Saved processed with folds to: artifacts/train_processed.csv


In [ ]:
# ==================== 5. Label transform (log1p or gram) & Save processing_config ====================
RAW_TARGETS = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]

# 1) Regardless of whether the current one is used or not, calculate the *_log column first (only once)
for t in RAW_TARGETS:
    neg_ct = (df_pivot[t] < 0).sum()
    if neg_ct > 0:
        print(f"[Warn] {t} has {neg_ct} negatives; clipping to 0 before log1p.")
        df_pivot.loc[df_pivot[t] < 0, t] = 0.0
        
for t in RAW_TARGETS:
    log_col = f"{t}_log"
    if log_col not in df_pivot.columns:
        df_pivot[log_col] = np.log1p(df_pivot[t].astype("float32"))
        
# ====================        
# 2) Define a helper to switch label mode (log1p vs gram)
def set_label_mode(use_log1p: bool):
    """
    Switch the target columns used during training:
        use_log1p=True → Use the *_log columns (log1p)
        use_log1p=False → Use the original gram columns 
    """
    global TRAIN_TARGETS, label_transform
    
    if not hasattr(CFG, 'use_log1p'):
        CFG.use_log1p = False
    
    CFG.use_log1p = use_log1p

    if use_log1p:
        TRAIN_TARGETS = [f"{t}_log" for t in CORE_TARGETS]
        label_transform = "log1p"
    else:
        TRAIN_TARGETS = CORE_TARGETS
        label_transform = "identity" # 'identity' means no transform / raw grams

    print(f"[set_label_mode] use_log1p={use_log1p} → TRAIN_TARGETS =", TRAIN_TARGETS)
    print(f"[init] fallback label_transform={label_transform}")
    
# 3) Set a default mode (e.g., gram) for preprocessing/config
set_label_mode(False)  # default set to gram
# ====================

# Overwrite save (contains both raw and *_log)
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)
print("Saved (with log columns):", processed_path)


# Assemble and save processing_config.json
processing_cfg = {
    "project_title": CFG.project_title,
    "seed": CFG.seed,
    "image_size_h": CFG.image_size_h,
    "image_size_w": CFG.image_size_w,
    "n_folds": CFG.n_folds,
    "batch_size": CFG.batch_size,
    "use_log1p": CFG.use_log1p,  # will reflect the last mode you set
    "label_transform": label_transform,
    "train_target_cols": TRAIN_TARGETS,         # train reads these columns
    "train_target_cols_raw": CORE_TARGETS,      # corresponding physical meanings
    "all_targets_raw": RAW_TARGETS,
    "artifacts_dir": CFG.artifacts_dir,
    "image_root": CFG.image_root,
    "n_samples": int(len(df_pivot)),
    "n_unique_images": int(df_pivot["image_id"].nunique()),
    "State_map": STATE_MAP,
    "Species_map": SPECIES_MAP,
}

# save json
proc_cfg_path = Path(CFG.artifacts_dir) / "processing_config.json"
proc_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(proc_cfg_path, "w", encoding="utf-8") as f:
    json.dump(processing_cfg, f, indent=2, ensure_ascii=False)

print("Saved processing config:", proc_cfg_path)

[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity
Saved (with log columns): artifacts/train_processed.csv
Saved processing config: artifacts/processing_config.json


In [ ]:
# ==================== 6. Config & Processed Data Saving ====================

from pathlib import Path
import json
from dataclasses import asdict
import numpy as np

# 1) Define the list of metadata features to be used
METADATA_FEATURES = [
    "Pre_GSHH_NDVI",      # Pre-season NDVI from satellite
    "Height_Ave_cm",      # Average plant height in cm
    "State_encoded",      # One-hot or label-encoded growth state
    "Species_encoded",    # Encoded plant species
    "year",               # Year of observation
    'month_sin',          # Cyclic encoding of month (sin)
    'month_cos',          # Cyclic encoding of month (cos)
    'day_of_year_sin',    # Cyclic encoding of day-of-year (sin)
    'day_of_year_cos',    # Cyclic encoding of day-of-year (cos)
]

# 2) Automatically separate categorical and continuous metadata columns
META_COLS_CAT  = [col for col in METADATA_FEATURES if 'encoded' in col]   # Categorical (already encoded)
META_COLS_CONT = [col for col in METADATA_FEATURES if col not in META_COLS_CAT]  # Continuous

# 3) Apply Z-score standardization to continuous metadata features
for col in META_COLS_CONT:
    mean_val = df_pivot[col].mean()
    std_val  = df_pivot[col].std()
    
    # Avoid division by zero or NaN
    if std_val == 0 or np.isnan(std_val):
        std_val = 1.0
    
    df_pivot[col] = (df_pivot[col] - mean_val) / std_val

print(f"Standardized continuous metadata features: {META_COLS_CONT}")
print(f"Categorical (encoded) metadata features: {META_COLS_CAT}")

# 4) Build configuration dictionary and save processed data
config_dict = {
    **asdict(CFG),                    # Include all settings from CFG dataclass
    "targets": CORE_TARGETS,          # List of prediction targets (e.g., biomass traits)
    "metadata_features": METADATA_FEATURES,
    "n_samples": len(df_pivot),
    "n_unique_images": df_pivot["image_id"].nunique(),
    "n_metadata_features": len(METADATA_FEATURES),
    "n_categorical_meta": len(META_COLS_CAT),
    "n_continuous_meta": len(META_COLS_CONT),
}

# Pretty-print the final config
print("\nFinal Configuration:")
print(json.dumps(config_dict, indent=4, default=str))

# 5) Save the processed training dataframe (with folds, normalized features, etc.)
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)



Standardized continuous metadata features: ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'year', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']
Categorical (encoded) metadata features: ['State_encoded', 'Species_encoded']

Final Configuration:
{
    "project_title": "CSIRO Image2Biomass Prediction",
    "seed": 42,
    "image_size_h": 224,
    "image_size_w": 448,
    "n_folds": 5,
    "batch_size": 8,
    "raw_csv": "csiro-biomass/train.csv",
    "image_root": "csiro-biomass",
    "artifacts_dir": "artifacts/",
    "figs_dir": "figs",
    "use_log1p": false,
    "targets": [
        "Dry_Green_g",
        "Dry_Dead_g",
        "Dry_Clover_g"
    ],
    "metadata_features": [
        "Pre_GSHH_NDVI",
        "Height_Ave_cm",
        "State_encoded",
        "Species_encoded",
        "year",
        "month_sin",
        "month_cos",
        "day_of_year_sin",
        "day_of_year_cos"
    ],
    "n_samples": 357,
    "n_unique_images": 357,
    "n_metadata_features": 9,
    "n_ca

## 2. images

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers

# ==================== 7. Augmentation primitives ====================

class ImageAugmentor(layers.Layer):
    """
    Custom Keras Layer for random image augmentation.
    
    Supports both:
      - Rank-3 inputs: (H, W, C)   → single image
      - Rank-4 inputs: (N, H, W, C) → batch of images
      
    Designed to be used:
      - Directly inside a Keras model (as a layer)
      - Or in tf.data pipelines via .map(tfdata_augmentor)
      
    Only augments the image (x), leaves labels (y) untouched.
    """
    
    def __init__(self, 
                 color_prob=0.15,
                #  gray_prob=0.0,
                 noise_prob=0.2,
                 blur_prob=0.2,
                 flip_lr_prob=0.5,
                 flip_ud_prob=0.5,
                 rot90_prob=0.5,
                 **kwargs):
        super().__init__(**kwargs)
        
        # augmentation config: (name, probability, apply function)
        self.ops = [
            ('Hue +0.02', color_prob, 
             lambda x: tf.image.adjust_hue(x, 0.02)),
            ('Saturation x1.1', color_prob, 
             lambda x: tf.image.adjust_saturation(x, 1.1)),
            ('Brightness +0.1', color_prob, 
             lambda x: tf.image.adjust_brightness(x, 0.1)),
            ('Contrast x1.1', color_prob, 
             lambda x: tf.image.adjust_contrast(x, 1.1)),
            # ('Grayscale', gray_prob, 
            #  lambda x: tf.image.grayscale_to_rgb(tf.image.rgb_to_grayscale(x))),
            ('Noise σ=0.02', noise_prob, 
             lambda x: x + tf.random.normal(tf.shape(x), stddev=0.02)),
            ('BoxBlur 3x3', blur_prob, 
             lambda x: tf.nn.avg_pool2d(x, 3, 1, 'SAME')),
            ('Flip LR', flip_lr_prob, 
             tf.image.flip_left_right),
            ('Flip UD', flip_ud_prob, 
             tf.image.flip_up_down),
            # ('Rotate 90°', rot90_prob, 
            #  lambda x: tf.image.rot90(x, k=1)),
        ]


    def get_visualization_ops(self, include_prob=False):
        """
        Helper to visualize what augmentations are applied (useful for debugging/logging).
        """
        if include_prob:
            return [(f"{name} ({prob:.0%})", fn) for name, prob, fn in self.ops]
        return [(name, fn) for name, prob, fn in self.ops]
    
    def call(self, x, training=None):
        """
        Keras Layer: g(x) -> x_aug
        - training=True/None: apply random augmentations
        - training=False: return img unchanged
        """
        # Handle single image (H,W,C) → temporarily add batch dim
        is_single = x.shape.rank == 3
        if is_single:
            x = x[None]  # (H,W,C) -> (1,H,W,C)
            
        # Default behavior in Keras: training=None means apply augmentations
        if training is None:
            training = True 
            
        # If training is a Tensor (e.g., traced in tf.data), use tf.cond to control overall logic
        def apply_all_ops(img):
            for _, prob, fn in self.ops:
                if prob > 0:
                    img = tf.cond(
                        tf.random.uniform(()) < prob,
                        lambda: tf.clip_by_value(fn(img), 0., 1.),
                        lambda: img,
                    )
            return img

        #  when training is a Tensor
        if isinstance(training, tf.Tensor):
            x = tf.cond(
                tf.cast(training, tf.bool),
                lambda: apply_all_ops(x),
                lambda: x,
            )
        elif training:
            x = apply_all_ops(x)
            
        # Remove batch dim if input was a single image
        return x[0] if is_single else x

# Instantiate the augmentor layer (can be reused)
img_aug_layer = ImageAugmentor(name="augmentor")

# def tfdata_augmentor(x, y): # f(x, y) → (x_aug, y)
#     return img_aug_layer(x, training=True), y


def tfdata_augmentor(x, y):
    """
    Universal augmentation wrapper for tf.data pipelines.
    
    Supports two input formats:
      - Image-only: x is a Tensor (H,W,C) or (N,H,W,C)
      - Hybrid model: x is a dict with key 'image'
      
    Returns: (x_augmented, y) — labels unchanged
    """
    # Case 1: x = (image, metadata)
    if isinstance(x, dict):
        x = dict(x)  # shallow copy to avoid mutating original
        x["image"] = img_aug_layer(x["image"], training=True)
        return x, y
    # Case 2: x = image only
    else:
        x_aug = img_aug_layer(x, training=True)
        return x_aug, y


def tfdata_clip(x, y): # Numerical Safety / Clamping
    """
    Universal clipping function (ensures pixel values stay in [0, 1]).
    Works with both image-only and dict-style inputs.
    """
    if isinstance(x, dict):
        x = dict(x)
        x["image"] = tf.clip_by_value(x["image"], 0.0, 1.0)
        return x, y
    else:
        return tf.clip_by_value(x, 0.0, 1.0), y


In [ ]:
# ==================== 8. BiomassImgDataModule  ====================

class BiomassImgDataModule:
    def __init__(self, 
                 fold: int = 3, 
                 target_list: list = None,
                 include_meta: bool = False):  # <--- 1. tabular switch: Teacher-True,  Student/Baseline-False
        """
        include_meta=True: return ({'image': img, 'meta_cont':..., 'meta_cat':...}, y) → to Hybrid Teacher
        
        → **Input (x)**: A dictionary containing the image and all tabular metadata.
        → **Output (y)**: The 3 core biomass targets (Dry_Green_g, Dry_Dead_g, Dry_Clover_g).
        
        ---------------------------------------------------------
        
        include_meta=False: return (img, y) → to Image-Only Student/Baseline
        
        → **Input (x)**: A tensor containing only the image (One image).
        → **Output (y)**: The 3 core biomass targets (Dry_Green_g, Dry_Dead_g, Dry_Clover_g).
        
        **Summary:** Each record (One sample) corresponds to 3 core target values (3 target).
        """
        self.fold = fold
        self.include_meta = include_meta       # <--- 2. include tabular
        
        if target_list is None:
            target_list = TRAIN_TARGETS
        self.target_list = target_list

        self.IMG_SIZE = (CFG.image_size_h, CFG.image_size_w)

        # --------------------- Load processed data ---------------------
        pivot_path = out_dir / "train_processed.csv"
        assert pivot_path.exists(), f"Missing: {pivot_path}"
        df = pd.read_csv(pivot_path)

        # Drop rows missing any target (critical for regression)
        df = df.dropna(subset=self.target_list).reset_index(drop=True)

        self.df = df
        self.y = df[self.target_list].astype("float32").values
        
        # --------------------- Metadata preprocessing (if used) ---------------------
        if self.include_meta:
            # 1. 
            self.meta_cols_cat  = META_COLS_CAT
            self.meta_cols_cont = META_COLS_CONT

            # Fill NaN in continuous metadata (common & safe default)
            self.df[self.meta_cols_cont] = self.df[self.meta_cols_cont].fillna(0.0)
            

    def load_img(self, path: str):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img = cv2.resize(
            img,
            (self.IMG_SIZE[1], self.IMG_SIZE[0]), 
            interpolation=cv2.INTER_AREA
        )
        return img.astype("float32") / 255.0

    def make_arrays(self):
        # imgs → N x H x W x C
        imgs = np.stack([self.load_img(p) for p in self.df["abs_image_path"]])
        return imgs

    def make_datasets(self):
        imgs = self.make_arrays()
        y = self.y # y → N x T

        val_mask = (self.df["fold"] == self.fold).values
        tr_idx = np.where(~val_mask)[0]
        va_idx = np.where(val_mask)[0]

        print(f"Fold {self.fold} | Target: {self.target_list}")
        print(f"  Train: {len(tr_idx)} images | Val: {len(va_idx)} images")

        # # tf.data datasets
        # train_ds = tf.data.Dataset.from_tensor_slices((imgs[tr_idx], y[tr_idx]))
        # val_ds   = tf.data.Dataset.from_tensor_slices((imgs[va_idx], y[va_idx]))
        
        # ==========================================
        # Core Logic: Choose input format based on whether metadata is used
        # ==========================================
        if self.include_meta:
            # Mode A: Hybrid (Teacher) -> Returns a dictionary of inputs
            # Useful for models that accept multiple inputs (image + tabular metadata)
            meta_cont = self.df[self.meta_cols_cont].astype("float32").values # (N, 7)
            meta_cat  = self.df[self.meta_cols_cat].astype("int32").values # (N, 2)
            
            # Dataset yields (dict_of_inputs, labels)
            train_inputs = {
                "image":     imgs[tr_idx],
                "meta_cont": meta_cont[tr_idx],
                "meta_cat":  meta_cat[tr_idx],
            }
            val_inputs = {
                "image":     imgs[va_idx],
                "meta_cont": meta_cont[va_idx],
                "meta_cat":  meta_cat[va_idx],
            }

            train_ds = tf.data.Dataset.from_tensor_slices((train_inputs, y[tr_idx]))
            val_ds   = tf.data.Dataset.from_tensor_slices((val_inputs,   y[va_idx]))

        else:
            # Mode B: Image Only (Baseline / Student) -> Returns Tensor
            # Simpler and faster - perfect for pure CNNs or knowledge distillation students
            train_ds = tf.data.Dataset.from_tensor_slices((imgs[tr_idx], y[tr_idx]))
            val_ds   = tf.data.Dataset.from_tensor_slices((imgs[va_idx], y[va_idx]))


        # ==========================================
        # Pipeline (for all model)
        # ==========================================
        print(f"Fold {self.fold} | Meta={self.include_meta} | Train:{len(tr_idx)} Val:{len(va_idx)}")
        
        # Create Image Data Pipeline
        train_ds = (
            train_ds
            .map(tfdata_augmentor, num_parallel_calls=1)# x passed to aug function, preprocessing + Augmentation
            .shuffle(1024, seed=CFG.seed, reshuffle_each_iteration=False) # Shuffle order
            .batch(CFG.batch_size)  # batching
            .prefetch(1) # prefetch for performance(speed up training, important!)
        )

        val_ds = (
            val_ds
            .map(tfdata_clip, num_parallel_calls=1)
            .batch(CFG.batch_size)
            .prefetch(1)
        )

        return train_ds, val_ds


In [ ]:
dm = BiomassImgDataModule(fold=1)
imgs = dm.make_arrays()
print(imgs.shape)   # should be (N, 112, 224, 3)

(357, 224, 448, 3)


## 3. cnn model

In [ ]:

# ==================== 9. Weighted R² metric  ====================
import numpy as np
import pandas as pd

COMP_WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5,
}

core_loss_weights = [
    COMP_WEIGHTS["Dry_Green_g"],   # 0.1
    COMP_WEIGHTS["Dry_Dead_g"],    # 0.1
    COMP_WEIGHTS["Dry_Clover_g"],  # 0.1
]

def weighted_r2_kaggle(df_true, df_pred):
    """
    df_true, df_pred: DataFrame, with columns:
      ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    All values are on the ORIGINAL scale (grams), no logs.
    """

    cols = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]

    y_list, yp_list, w_list = [], [], []

    for col in cols:
        y  = df_true[col].astype("float64").values
        yp = df_pred[col].astype("float64").values
        w  = np.full_like(y, fill_value=COMP_WEIGHTS[col], dtype="float64")

        y_list.append(y)
        yp_list.append(yp)
        w_list.append(w)

    y_all  = np.concatenate(y_list)      # all (image, target) in one array
    yp_all = np.concatenate(yp_list)
    w_all  = np.concatenate(w_list)

    # Global weighted average
    y_bar_w = np.sum(w_all * y_all) / np.sum(w_all)

    ss_res = np.sum(w_all * (y_all - yp_all)**2)
    ss_tot = np.sum(w_all * (y_all - y_bar_w)**2)

    r2w = 1.0 - ss_res / (ss_tot + 1e-12)
    return float(r2w)
      

def weighted_mse_loss(weights, max_err=None):
    """Create a weighted MSE loss function.

    Args:
        weights: 1D sequence of per-target weights.
        max_err: Optional max absolute error for clipping. If None, no clipping.

    Returns:
        Callable: Loss function `loss_fn(y_true, y_pred)` for Keras.
    """
    w = tf.constant(weights, tf.float32)

    def loss_fn(y_true, y_pred):
        err = y_true - y_pred  # (B, T)
        if max_err is not None:
            # Clip the absolute value of individual errors to prevent explosion
            err = tf.clip_by_value(err, -max_err, max_err)
        err2 = tf.square(err)
        return tf.reduce_mean(err2 * w)
    return loss_fn

def eval_weighted_r2_in_gram(
        val_df: pd.DataFrame,
        y_pred_default: np.ndarray,
        model_name: str = "",
        fold: int | None = None):
    """
    Evaluate model predictions using Kaggle-style Weighted R² in gram space.

    Args:
        val_df (DataFrame): Ground-truth labels in gram units.
        y_pred_default (ndarray): Model outputs (log1p or gram), shape (N, 3).
        model_name (str, optional): Name for logging.
        fold (int, optional): Fold index for logging.

    Returns:
        tuple: (weighted_r2, pred_df, true_df)
    """
    # 1) Convert model outputs to gram space based on the current label mode
    if CFG.use_log1p:
        # default outputs are log1p → convert back to grams
        y_pred_gram = np.expm1(y_pred_default)
    else:
        # default outputs are already in grams
        y_pred_gram = y_pred_default

    # 2) Extract the three core targets
    dg_p = y_pred_gram[:, 0]
    dd_p = y_pred_gram[:, 1]
    dc_p = y_pred_gram[:, 2]

    # 3) Construct prediction DataFrame (including derived GDM / Dry_Total)
    pred_df = pd.DataFrame({
        "Dry_Green_g":  dg_p,
        "Dry_Dead_g":   dd_p,
        "Dry_Clover_g": dc_p,
    })
    pred_df["GDM_g"]       = pred_df["Dry_Green_g"] + pred_df["Dry_Clover_g"]
    pred_df["Dry_Total_g"] = pred_df["Dry_Green_g"] + pred_df["Dry_Dead_g"] + pred_df["Dry_Clover_g"]

    # 4) Ground truth (always in gram space)
    true_df = val_df[[
        "Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"
    ]].copy()

    # 5) Compute Weighted R²
    w_r2 = weighted_r2_kaggle(true_df, pred_df)

    tag = f"[{model_name}] " if model_name else ""
    if fold is not None:
        print(f"{tag}Fold {fold} Kaggle-style weighted R2 = {w_r2:.4f}")
    else:
        print(f"{tag}Kaggle-style weighted R2 = {w_r2:.4f}")

    return w_r2, pred_df, true_df

In [ ]:
# ==================== 10. backbone ====================
# this model, it uses ONLY pure image data — no tabular features at all!

from tensorflow.keras import layers, models, optimizers, callbacks
def make_mae_metrics(target_names):
    """
    Create overall MAE metric + per-target MAE metrics for Keras model compilation.
    
    Supports two scenarios:
    - Normal training: y_true shape = (B, T)
    - Distillation training: y_true shape = (B, 2T), 
      where the first T dimensions are ground truth labels,
      and the latter T dimensions are teacher predictions.
    """
    n_targets = len(target_names)

    def mae_all(y_true, y_pred):
        # Only evaluate on the ground truth part (first half)
        y_true_core = y_true[:, :n_targets]  
        return tf.reduce_mean(tf.abs(y_true_core - y_pred))

    mae_all.__name__ = "mae_all"
    metric_list = [mae_all]

    def make_fn(idx, name):
        def mae_i(y_true, y_pred):
            y_true_core = y_true[:, :n_targets]
            return tf.reduce_mean(tf.abs(y_true_core[:, idx] - y_pred[:, idx]))
        mae_i.__name__ = name
        return mae_i

    for i, tname in enumerate(target_names):
        metric_list.append(make_fn(i, f"mae_{tname}"))
    return metric_list

# === shared image backbone ===
def build_image_backbone(
    image_size_h: int = CFG.image_size_h,
    image_size_w: int = CFG.image_size_w,
    name: str = "ImageBackbone",
):
    """Build a basic CNN backbone for image feature extraction.

    Args:
        image_size_h: Input image height in pixels.
        image_size_w: Input image width in pixels.

    Returns:
        tf.keras.Model: CNN backbone mapping (H, W, 3) → (batch, 256) features.
    """
    
    img_shape = (image_size_h, image_size_w, 3)
    inputs = layers.Input(shape=img_shape)

    x = layers.Conv2D(16, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 112x224

    x = layers.Conv2D(32, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 56x112

    x = layers.Conv2D(64, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 28x56

    x = layers.Conv2D(256, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)   # -> (batch, 256)

    backbone = models.Model(inputs=inputs, outputs=x, name=name)
    return backbone

# ==================== 11A. BiomassConvBaseline (multi-target CNN regressor) ====================
def biomass_conv_baseline(
   image_size_h: int = CFG.image_size_h, 
   image_size_w: int = CFG.image_size_w, 
   target_names: list | None = None,
) -> tf.keras.Model:
   """
   Simple CNN baseline for multi-target biomass regression.
   Input : image (H,W,3)
   Output: len(target_names) regression values.
   """
   if target_names is None:
       target_names = TRAIN_TARGETS  # default to all targets
   n_targets = len(target_names)

   # Shared image backbone, as a layer for any model
   backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackbone")
   img_inputs = layers.Input(shape=(image_size_h, image_size_w, 3), name="image")
   x = backbone(img_inputs)
  
   # Head: regression part
   x = layers.Dense(128, activation="relu")(x)
   x = layers.Dropout(0.3)(x)
   outputs = layers.Dense(
       n_targets,
       activation="softplus", # Softplus to ensure non-negative outputs
       name="biomass",
       dtype="float32",
   )(x)

   model = models.Model(inputs=img_inputs, outputs=outputs,
                        name="BiomassConvBaseline")
   
    # Loss weights for the three core competition targets
   core_loss_weights = [
       COMP_WEIGHTS["Dry_Green_g"],
       COMP_WEIGHTS["Dry_Dead_g"],
       COMP_WEIGHTS["Dry_Clover_g"],
   ]

   model.compile(
       optimizer=optimizers.Adam(1e-3),
       loss=weighted_mse_loss(core_loss_weights), # Custom weighted MSE (assumed defined elsewhere)
       metrics=make_mae_metrics(target_names),
   )

   return model


In [ ]:
# ==================== 11.0. refers: true Mean baseline ≈ 0.24  ====================
def mean_baseline_r2_for_fold(fold: int):
    """
    Compute weighted R² for a per-fold mean target baseline.

    For the given fold, this baseline predicts the mean of each target
    (computed on the training split) for all validation samples, then
    evaluates Kaggle-style weighted R².

    Args:
        fold: Fold index to use as validation split.

    Returns:
        float: Weighted R² score for this fold.
    """
    dm = BiomassImgDataModule(fold=fold, target_list=TRAIN_TARGETS)

    val_mask   = (dm.df["fold"] == fold).values
    train_mask = ~val_mask

    train_df = dm.df[train_mask].reset_index(drop=True)
    val_df   = dm.df[val_mask].reset_index(drop=True)

    # mean of train_df (gram)
    mean_df = train_df[["Dry_Green_g","Dry_Dead_g","Dry_Clover_g"]].mean()

    n_val = len(val_df)
    pred_df = pd.DataFrame({
        "Dry_Green_g":  np.full(n_val, mean_df["Dry_Green_g"]),
        "Dry_Dead_g":   np.full(n_val, mean_df["Dry_Dead_g"]),
        "Dry_Clover_g": np.full(n_val, mean_df["Dry_Clover_g"]),
    })
    pred_df["GDM_g"]       = pred_df["Dry_Green_g"] + pred_df["Dry_Clover_g"]
    pred_df["Dry_Total_g"] = pred_df["Dry_Green_g"] + pred_df["Dry_Dead_g"] + pred_df["Dry_Clover_g"]

    true_df = val_df[[
        "Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"
    ]].copy()

    r2 = weighted_r2_kaggle(true_df, pred_df)
    print(f"[Mean baseline] Fold {fold}: weighted R² = {r2:.4f}")
    return r2

mean_r2 = []
for f in range(CFG.n_folds):
    mean_r2.append(mean_baseline_r2_for_fold(f))

print("Mean-baseline R² per fold:", np.round(mean_r2, 3),
      "mean =", np.mean(mean_r2))


[Mean baseline] Fold 0: weighted R² = 0.2467
[Mean baseline] Fold 1: weighted R² = 0.2703
[Mean baseline] Fold 2: weighted R² = 0.2077
[Mean baseline] Fold 3: weighted R² = 0.2795
[Mean baseline] Fold 4: weighted R² = 0.2122
Mean-baseline R² per fold: [0.247 0.27  0.208 0.279 0.212] mean = 0.24327056587010784


In [ ]:


# ==================== 11. one fold train generic model ====================

from tensorflow.keras import callbacks

def build_callbacks(monitor, mode):
    """Create standard training callbacks.

    Args:
        monitor: Metric name to monitor (e.g. "val_mae_all").
        mode: "min" or "max", passed to Keras callbacks.

    Returns:
        list: [EarlyStopping, ReduceLROnPlateau] callbacks.
    """
    rlr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        mode=mode,                
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    )
    es = callbacks.EarlyStopping(
        monitor=monitor,
        mode=mode,
        patience=5,
        restore_best_weights=True,
        verbose=1,
    )
    return [es, rlr] # callbacks_list, early_stopping


def build_val_inputs(
    dm: BiomassImgDataModule,
    fold: int,
    include_meta: bool,
):
    """Build validation inputs for R² evaluation.

    Args:
        dm: Data module holding processed DataFrame and image arrays.
        fold: Fold index to select validation rows.
        include_meta: If True, return dict(image + metadata); otherwise images only.

    Returns:
        tuple:
            - val_df (pd.DataFrame): Validation rows in gram space.
            - x_val: Validation inputs for model.predict (tensor or dict).
    """
    # 1) Select the current fold's val df (with 5 real targets in GRAM space)
    val_mask = (dm.df["fold"] == fold).values
    val_df   = dm.df[val_mask].reset_index(drop=True)

    # 2) prepare val img array
    imgs_all = dm.make_arrays()              # (N, H, W, 3)
    imgs_val = imgs_all[val_mask]            # (N_val, H, W, 3)

    # 3)Decide the structure of x_val based on include_meta
    # include_meta=False → image-only pipeline + x_val = imgs_val
    # include_meta=False → image-only pipeline + x_val = imgs_val
    # CFG.use_log1p control log/gram
    if include_meta:
        # Teacher: image + tabular dict
        meta_cont_all = dm.df[dm.meta_cols_cont].astype("float32").values
        meta_cat_all  = dm.df[dm.meta_cols_cat ].astype("int32").values

        meta_cont_val = meta_cont_all[val_mask]
        meta_cat_val  = meta_cat_all[val_mask]

        x_val = {
            "image":     imgs_val,
            "meta_cont": meta_cont_val,
            "meta_cat":  meta_cat_val,
        }
    else:
        # Baseline: image only
        x_val = imgs_val
    return val_df, x_val


def train_one_fold_generic(
    fold,
    *,
    include_meta: bool,
    model_fn,
    model_name: str,
    target_list=None,
    **model_kwargs, #  baseline_weights_path=baseline_ckpt pass to model_fn
):
    """
    Train one fold.

    If target_list is None, default to TRAIN_TARGETS.
    When using log1p, TRAIN_TARGETS should be the *_log names if CFG.use_log1p is True.
    Args:
        fold: which fold index to use as validation.
        target_list: list of target column names; default = TRAIN_TARGETS.
        model_fn: a function that builds and returns a compiled Keras model.
                  Signature: model_fn(image_size_h, image_size_w, target_names, **model_kwargs)
        model_kwargs: extra keyword args passed to model_fn (e.g. backbone_trainable_ratio).
    Returns:
        tuple: (best_val_mae, model, history, val_ds, w_r2)
    """
    
    if target_list is None:
        target_list = TRAIN_TARGETS

    print(f"\n===== Fold {fold} =====")
    print(f"[Baseline] use_log1p={CFG.use_log1p}, target_list={target_list}")

    # ---- Choose a fold for validation and train ----
    dm = BiomassImgDataModule(
        fold=fold, 
        target_list=target_list,
        include_meta=include_meta,   # <--- baseline exclude tabular,image only
    )
    train_ds, val_ds = dm.make_datasets()
  
    # ---- create model ----
    model = model_fn( # 9A.biomass_conv_baseline
        image_size_h = CFG.image_size_h, 
        image_size_w = CFG.image_size_w,
        target_names=target_list,
        **model_kwargs,
    )
  
    # ---- callbacks ----
    callbacks_list = build_callbacks(monitor="val_mae_all", mode="min")
    
    # ---- Train fold ----
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        callbacks=callbacks_list,
        verbose=1,
    )

    # ---- Evaluate fold ----
    #  Get best val_mae_all from 5 folds history
    best_val_mae = min(history.history["val_mae_all"])
    print(f"Fold {fold} best val_mae_all = {best_val_mae:.3f}")
    
    
    # =========  Weighted R² (eval ALWAYS in gram)  ========
    # build_val_inputs + eval_weighted_r2_in_gram
    
    # 1) Construct val_df + x_val (image-only or image+meta)
    val_df, x_val = build_val_inputs(
        dm=dm,
        fold=fold,
        include_meta=include_meta,
    )

    # 2) model predict (use gram to caculate R2), if its log1p, then convert to grams if needed
    y_pred_default= model.predict(x_val, batch_size=CFG.batch_size)
                        
    # 3) Calculate R² in gram
    w_r2, pred_df, true_df = eval_weighted_r2_in_gram(
        val_df=val_df,
        y_pred_default=y_pred_default,
        model_name=model_name,
        fold=fold,
    )
    # ========================================
    return best_val_mae, model, history, val_ds, w_r2,

In [ ]:
import os
os.makedirs("./ckpt", exist_ok=True)
# wrappers 
def train_one_fold_baseline(
    fold,
    target_list=None,
    model_fn=biomass_conv_baseline,
    save_ckpt: bool = False,   # 👈 
    **model_kwargs,
):
    """
    Baseline CNN: image-only model (no metadata).
    return: best_val_mae, model, history, val_ds, w_r2
    option to save fold's weight to ./ckpt/
    """
    # Use the generic fold training pipeline
    best_val_mae, model, history, val_ds, w_r2 = train_one_fold_generic(
        fold=fold,
        model_fn=model_fn,
        include_meta=False,     # image-only baseline
        target_list=target_list,   # default = TRAIN_TARGETS via set_label_mode
        model_name=f"baseline_fold{fold}",
        **model_kwargs,
    )

    # Optionally save checkpoint for initializing student later
    if save_ckpt:
        mode = "log1p" if CFG.use_log1p else "gram"  
        ckpt_path = f"./ckpt/baseline_{mode}_fold{fold}.weights.h5"
        model.save_weights(ckpt_path)
        print(f"[Baseline] fold {fold} weights saved to {ckpt_path}")

    print(f"[Baseline] fold {fold} best val_mae_all = {best_val_mae:.3f}, "
          f"weighted R² = {w_r2:.3f}")
    return best_val_mae, model, history, val_ds, w_r2


In [ ]:

# ==================== 11B. train CNN Baseline (gram) ====================
set_label_mode(False)  # use gram columns as training target
include_meta: bool = False # <--- 1. tabular switch, image only for baseline

# run 5-fold CV
baseline_fold_maes_gram = []
baseline_models_gram = []
baseline_val_ds_gram = []
baseline_w_r2_gram = []

# 🔴 give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):  # CFG.n_folds = 5
# for f in [1]:               # [x] = specific fold number you want to run 
    mae, m, history, val_ds, w_r2 = train_one_fold_baseline(f, save_ckpt=True) # stop passing target_list, instead default to TRAIN_TARGETS
    baseline_fold_maes_gram.append(mae)
    baseline_models_gram.append(m)
    baseline_val_ds_gram.append(val_ds)
    baseline_w_r2_gram.append(w_r2)

baseline_fold_maes_gram = np.array(baseline_fold_maes_gram)
print("\n===== [CNN Baseline - gram] CV summary =====")
print("val_mae_all per fold:", np.round(baseline_fold_maes_gram, 3))
print("mean ± std =", baseline_fold_maes_gram.mean(), "±", baseline_fold_maes_gram.std())
print("weighted_r2 per fold:", np.round(baseline_w_r2_gram, 3))
print("basline_gram mean R2 =", np.mean(baseline_w_r2_gram))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=False | Train:285 Val:72
Epoch 1/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 750ms/step - loss: 46.1078 - mae__dry__clover_g: 7.0151 - mae__dry__dead_g: 10.1789 - mae__dry__green_g: 20.5976 - mae_all: 12.5972 - val_loss: 164.6895 - val_mae__dry__clover_g: 19.9228 - val_mae__dry__dead_g: 14.7816 - val_mae__dry__green_g: 62.0170 - val_mae_all: 32.2404 - learning_rate: 0.0010
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 746ms/step - loss: 29.7379 - mae__dry__clover_g: 8.0229 - mae__dry__dead_g: 9.3600 - mae__dry__green_g: 16.1116 - mae_all: 11.1648 - val_loss: 90.9860 - val_mae__dry__clover_g: 9.8426 - val_mae__dry__dead_g: 14.1329 - val_mae__dry__gr

In [ ]:
# only to check model structure summary

baseline_model = biomass_conv_baseline( 
    image_size_h=CFG.image_size_h,
    image_size_w=CFG.image_size_w,
    target_names=CORE_TARGETS,
)
baseline_model.summary()


Model: "BiomassConvBaseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 448, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ImageBackbone (Functional)      │ (None, 256)            │       172,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ biomass (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,051 (804.89 KB)

 Trainable params: 205,315 (802.01 KB)

 Non-trainable params: 736 (2.88 KB)

____________________________________________________________________________________________________________________________________

____________________________________________________________________________________________________________________________________

Advanced Multi-Modal Biomass Prediction with Knowledge Distillation
====================================================================

Architecture Hierarchy:
1. Hybrid Teacher (Multi-Modal) - uses ALL information
2. Meta-Aware Hybrid Teacher (Enhanced) - with attention mechanisms  
3. Distilled Student (Image-Only) - deployable model
4. Ensemble Strategy - combining multiple approaches


This module implements:
✓ Hybrid Teacher with cross-modal attention
✓ Knowledge distillation framework
✓ Image-only student for deployment
✓ Progressive training pipeline
✓ Ensemble strategies

$$x = \left\{
\begin{array}{ll}
\text{"image"}: & \text{Image Tensor} \\
\text{"meta\_cont"}: & \text{Continuous Features Tensor} \\
\text{"meta\_cat"}: & \text{Categorical Features Tensor}
\end{array}
\right\}$$

In [ ]:
# ==================== 12.1. Hybrid Teacher model (Dict Input-key for Multi-Modal Fusion)====================

from tensorflow.keras import models
from tensorflow.keras import layers, models, optimizers, callbacks
# Hybrid Teacher model:
# image_backbone → 256-d tensor
# meta_cont → Dense(32/64) ×2
# meta_cat → Embedding(4+4/8+8) → concat
# concat → Dense(64/128) → Dropout → Dense(3)

def biomass_hybrid_teacher(
    image_size_h: int = CFG.image_size_h,
    image_size_w: int = CFG.image_size_w,
    target_names: list | None = None,
) -> tf.keras.Model:
    """
    Hybrid Teacher:
      Input x: dict
        - x["image"]:     (H, W, 3) Image tensor.
        - x["meta_cont"]: Continuous metadata features (e.g., NDVI, Height, sine/cosine dates).
        - x["meta_cat"]:  Categorical metadata encoded [State_encoded, Species_encoded].
      Output: len(target_names) core targets (default 3).
    """
    if target_names is None:
        target_names = TRAIN_TARGETS
    n_targets = len(target_names)

    # Dictionary Inputs (Defining the expected inputs for the Keras Functional API)
    inputs = {
        "image":     layers.Input(shape=(image_size_h, image_size_w, 3), name="image"),
        "meta_cont": layers.Input(shape=(len(META_COLS_CONT),),  name="meta_cont"),
        "meta_cat":  layers.Input(shape=(len(META_COLS_CAT),), dtype="int32", name="meta_cat"),
    }

    # 1) image backbone
    # This block uses the predefined CNN structure (assumed to be similar to BiomassConvBaseline).
    backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackboneTeacher")
    img_feat = backbone(inputs["image"])   # Output: (Batch, 256) feature vector

    # 2) Continuous Features MLP Branch
    # Processes numerical (continuous) metadata features like NDVI and Height.
    cont_x = layers.Dense(64, activation="relu")(inputs["meta_cont"])  # 32->64
    cont_x = layers.BatchNormalization()(cont_x) 
    cont_x = layers.Dense(64, activation="relu")(cont_x)  # 32->64
    cont_x = layers.Dropout(0.2)(cont_x) 

    # 3) Categorical Features Embedding Branch
    # Processes encoded categorical features (State and Species).
    cat_inputs = inputs["meta_cat"]  # Input tensor shape is (batch, 2) 
    
    # Slicing the tensor to feed individual columns into separate embedding layers
    # defaultlt META_COLS_CAT = ["State_encoded", "Species_encoded"] 
    state_ids   = layers.Lambda(lambda x: x[:, 0])(cat_inputs) # State_encoded->outcome 0, 1, 2, ..., n-1 ontinuous integers
    species_ids = layers.Lambda(lambda x: x[:, 1])(cat_inputs) # Species_encoded
    
    # Define embedding dimensions
    state_emb_dim   = 8 
    species_emb_dim = 8 
    
    # Create and apply Embedding layers (converts integer codes to dense vectors)
    state_emb_layer   = layers.Embedding(len(STATE_MAP),   state_emb_dim,   name="state_emb") 
    species_emb_layer = layers.Embedding(len(SPECIES_MAP), species_emb_dim, name="species_emb")

    state_emb   = state_emb_layer(state_ids)     # Output:(batch, 8)
    species_emb = species_emb_layer(species_ids) # Output:(batch, 8)
    
    # Combine all categorical embeddings
    cat_emb = layers.Concatenate(name="cat_emb")([state_emb, species_emb])  # Output: (batch, 16)
    
    # Combine Continuous MLP output and Categorical Embeddings to form Tabular Feature Vector
    tab_x = layers.Concatenate(name="tabular_fused")([cont_x, cat_emb])

    # 4) Image + Tabular Fusion & Regression Head
    # Concatenate the high-level image features with the processed tabular features.
    fused = layers.Concatenate(name="img_tab_fusion")([img_feat, tab_x])
    
    # Final regression head MLP
    x = layers.Dense(128, activation="relu")(fused)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(
        n_targets,
        activation="softplus", # Softplus ensures non-negative biomass predictions - softplus(z) = log(1 + e^z)
        name="biomass",
        dtype="float32",
    )(x)
    
    # Construct the model
    model = models.Model(inputs=inputs, outputs=outputs, name="HybridTeacher")
    
    # 5) Compile Model
    # Reuse existing weighted loss function and MAE metrics.
    core_loss_weights = [
        COMP_WEIGHTS["Dry_Green_g"],
        COMP_WEIGHTS["Dry_Dead_g"],
        COMP_WEIGHTS["Dry_Clover_g"],
    ]

    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss=weighted_mse_loss(core_loss_weights, max_err=200.0),
        metrics=make_mae_metrics(target_names),
    )
    return model


In [ ]:

def train_one_fold_teacher(
    fold,
    target_list=None,
    model_fn=biomass_hybrid_teacher,
    **model_kwargs,
):
    """
    Hybrid Teacher: image + metadata model.
    """
    return train_one_fold_generic(
        fold=fold,
        model_fn=model_fn,
        include_meta=True,      # image + tabular
        target_list=target_list,
        model_name="Teacher",
        **model_kwargs,
    )

In [ ]:

# ====== run model: Hybrid Teacher (gram)======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

teacher_fold_maes_gram = []
teacher_models_gram = []
teacher_val_ds_gram = []
teacher_w_r2_gram = []

# 🔴 give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):
# for f in [1]:
    mae, m, history, val_ds, w_r2 = train_one_fold_teacher(f)
    teacher_fold_maes_gram.append(mae)
    teacher_models_gram.append(m)
    teacher_val_ds_gram.append(val_ds)
    teacher_w_r2_gram.append(w_r2)

teacher_fold_maes_gram = np.array(teacher_fold_maes_gram)
print("\n===== [Teacher] 5-fold CV summary =====")
print("val_mae_all per fold:", np.round(teacher_fold_maes_gram, 3))
print("mean ± std =", teacher_fold_maes_gram.mean(), "±", teacher_fold_maes_gram.std())
print("weighted_r2 per fold:", np.round(teacher_w_r2_gram, 3))
print("teacher mean R2 =", np.mean(teacher_w_r2_gram))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=True | Train:285 Val:72
Epoch 1/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 770ms/step - loss: 38.8546 - mae__dry__clover_g: 7.0409 - mae__dry__dead_g: 9.6411 - mae__dry__green_g: 18.8517 - mae_all: 11.8446 - val_loss: 107.6678 - val_mae__dry__clover_g: 11.6085 - val_mae__dry__dead_g: 16.0407 - val_mae__dry__green_g: 49.3199 - val_mae_all: 25.6564 - learning_rate: 0.0010
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 22.6278 - mae__dry__clover_g: 7.7688 - mae__dry__dead_g: 8.7898 - mae__dry__green_g: 14.1461 - mae_all: 10.2349 - val_loss: 33.0155 - val_mae__dry__clover_g: 6.2975 - val_mae__dry__dead_g: 9.9877 - val_mae__dry__green

In [ ]:
# only to check model structure summary

teacher_model = biomass_hybrid_teacher( 
    image_size_h=CFG.image_size_h,
    image_size_w=CFG.image_size_w,
    target_names=CORE_TARGETS,
)
teacher_model.summary()

Model: "HybridTeacher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ meta_cont           │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 64)        │        512 │ meta_cont[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meta_cat            │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_21[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_10 (Lambda)  │ (None)            │          0 │ meta_cat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_11 (Lambda)  │ (None)            │          0 │ meta_cat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 64)        │      4,160 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state_emb           │ (None, 8)         │         32 │ lambda_10[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ species_emb         │ (None, 8)         │        120 │ lambda_11[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 224, 448,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 64)        │          0 │ dense_22[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_emb             │ (None, 16)        │          0 │ state_emb[0][0],  │
│ (Concatenate)       │                   │            │ species_emb[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ImageBackboneTeach… │ (None, 256)       │    172,768 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tabular_fused       │ (None, 80)        │          0 │ dropout_16[0][0], │
│ (Concatenate)       │                   │            │ cat_emb[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_tab_fusion      │ (None, 336)       │          0 │ ImageBackboneTea… │
│ (Concatenate)       │                   │            │ tabular_fused[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 128)       │     43,136 │ img_tab_fusion[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 128)       │          0 │ dense_23[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ biomass (Dense)     │ (None, 3)         │        387 │ dropout_17[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 221,371 (864.73 KB)

 Trainable params: 220,507 (861.36 KB)

 Non-trainable params: 864 (3.38 KB)

____________________________________________________________________________________________________________________________________

In [ ]:

# ==================== 12.2. Student model ====================
# Distillation Loss for Student(image-only)
def distillation_loss(
    alpha=0.4,      # weight for teacher loss  alpha ∈ {0.2, 0.3, 0.5}  
    tau=1.0,        # temperature for distillation
    n_targets=len(CORE_TARGETS),
):
    """Create a distillation loss combining GT and teacher predictions.

    Args:
        alpha: Weight of the teacher term in the final loss
            (0 → pure GT, 1 → pure teacher).
        tau: Temperature used to soften student/teacher outputs.
        n_targets: Number of target dimensions.

    Returns:
        Callable: Loss function `loss_fn(y_true_concat, y_pred)` where
        `y_true_concat = [y_gt, y_teacher]` along the last dimension.
        
    Note:
    mse_distill ≈ (1 / tau**2) * MSE(y_pred, y_teacher)
    So the effective weights are:
    Loss ≈ (1 - alpha) * L_gt + alpha / (tau**2) * L_teacher
    """
    core_weights = tf.constant([
        COMP_WEIGHTS["Dry_Green_g"],
        COMP_WEIGHTS["Dry_Dead_g"],
        COMP_WEIGHTS["Dry_Clover_g"],
    ], tf.float32)

    def loss_fn(y_true_concat, y_pred):
        # Split ground truth and teacher predictions
        y_true    = y_true_concat[:, :n_targets]
        y_teacher = y_true_concat[:, n_targets:]
        
        # Student vs GT weighted MSE
        err_gt = tf.square(y_pred - y_true)
        mse_gt = tf.reduce_mean(err_gt * core_weights, axis=-1) 
        
        # Student vs teacher weighted MSE (with temperature)
        y_pred_t    = y_pred    / tau
        y_teacher_t = y_teacher / tau
        err_distill = tf.square(y_pred_t - y_teacher_t)
        mse_distill = tf.reduce_mean(err_distill * core_weights, axis=-1) 
        
        # Total loss = (1 - alpha) * GT + alpha * teacher
        return (1.0 - alpha) * mse_gt + alpha * mse_distill

    return loss_fn



In [ ]:
def biomass_student_from_baseline(
    image_size_h: int = CFG.image_size_h, 
    image_size_w: int = CFG.image_size_w, 
    target_names: list | None = None,
    baseline_weights_path: str | None = None,
    distill_alpha: float = 0.4,
     distill_tau: float = 1.0, 
    use_distill_loss: bool = True,
) -> tf.keras.Model:
    """Build the student CNN model initialized from the baseline backbone.

    The student is an image-only CNN. Optionally, it:
      * Loads backbone weights from a pre-trained baseline model.
      * Uses a distillation loss that mixes GT and teacher predictions.

    Args:
        image_size_h: Input image height in pixels.
        image_size_w: Input image width in pixels.
        target_names: List of target column names; if None, defaults to TRAIN_TARGETS.
        baseline_weights_path: Optional path to baseline model weights (.h5). If
            provided, the student backbone is initialized from these weights.
        distill_alpha: Weight of the teacher term in the distillation loss.
        distill_tau: Temperature for the distillation loss.
        use_distill_loss: If True, use distillation loss; otherwise, use
            weighted MSE on ground-truth only.

    Returns:
        tf.keras.Model: Compiled student model ready for training.
    """
    if target_names is None:
        target_names = TRAIN_TARGETS
    n_targets = len(target_names)

    # 1) Student backbone (image-only)
    backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackboneStudent")
    img_inputs = layers.Input(shape=(image_size_h, image_size_w, 3), name="image")
    x = backbone(img_inputs)

    # 2) Regression head (can mirror baseline or be slightly deeper)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    # Optionally add another layer:
    # x = layers.Dense(64, activation="relu")(x)
    # x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(
        n_targets,
        activation="softplus",
        name="biomass",
        dtype="float32",
    )(x)

    model = models.Model(
        inputs=img_inputs, outputs=outputs, name="StudentFromBaseline"
    )

    # 3) If baseline weights are provided, copy its backbone weights
    if baseline_weights_path is not None:
        print(f"[Student] load baseline backbone from {baseline_weights_path}")
        # Create a temporary baseline model to load weights into
        baseline_tmp = biomass_conv_baseline(
            image_size_h=image_size_h,
            image_size_w=image_size_w,
            target_names=target_names,
        )
        baseline_tmp.load_weights(baseline_weights_path)

        # Extract backbones
        baseline_backbone = baseline_tmp.get_layer("ImageBackbone")
        student_backbone  = model.get_layer("ImageBackboneStudent")

         # Copy CNN backbone parameters
        student_backbone.set_weights(baseline_backbone.get_weights())
        print("[Student] backbone weights copied from baseline")

        # Free the temporary model
        del baseline_tmp

     # 4) Choose loss: distillation vs pure GT (weighted MSE)
    core_loss_weights = [
        COMP_WEIGHTS["Dry_Green_g"],
        COMP_WEIGHTS["Dry_Dead_g"],
        COMP_WEIGHTS["Dry_Clover_g"],
    ]

    if use_distill_loss:
        # this alpha = “teacher w”
        loss_fn = distillation_loss(
            alpha=distill_alpha,
            tau=distill_tau,  
            n_targets=len(target_names),
        )
    else:
        loss_fn = weighted_mse_loss(core_loss_weights, max_err=200.0)

    model.compile(
        optimizer=optimizers.Adam(1e-4),   # Finetuning LR is smaller than baseline
        loss=loss_fn,
        metrics=make_mae_metrics(target_names),
    )
    return model


In [ ]:
def run_one_fold_student_from_baseline_no_distill(fold):
    """Train a student initialized from baseline backbone without distillation.

    The student backbone is initialized from the baseline checkpoint and then
    trained using only ground-truth supervision (no distillation loss).

    Args:
        fold: Fold index used as validation split.

    Returns:
        tuple: (best_val_mae, model, history, val_ds, w_r2)
    """
    baseline_ckpt = f"./ckpt/baseline_gram_fold{fold}.weights.h5"
    
    best_val_mae, model, history, val_ds, w_r2 = train_one_fold_generic(
        fold=fold,
        include_meta=False,   # student is also image-only
        model_fn=biomass_student_from_baseline,
        model_name=f"student_from_baseline_fold{fold}",
        baseline_weights_path=baseline_ckpt,
        distill_alpha=0.0,
        use_distill_loss=False,   # no distillation/teacher loss here
    )
    return best_val_mae, model, history, val_ds, w_r2


In [ ]:
# ====== Train model: Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram_no_distill = {}
student_fold_maes_gram_no_distill = []
student_fold_r2_gram_no_distill = []

# 🔴 give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        run_one_fold_student_from_baseline_no_distill(
            fold=f,
            # teacher_model=teacher_models_gram[f],   # trained teacher
            # alpha=3,
            # tau=5.0,
        )
    student_results_gram_no_distill[f] = (student_mae, student_r2)
    student_fold_maes_gram_no_distill.append(student_mae)
    student_fold_r2_gram_no_distill.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram_no_distill, 3))
print("R2  per fold:", np.round(student_fold_r2_gram_no_distill, 3))
print("mean R2 =", np.mean(student_fold_r2_gram_no_distill))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=False | Train:285 Val:72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/15


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 749ms/step - loss: 56.4363 - mae__dry__clover_g: 6.4797 - mae__dry__dead_g: 10.9026 - mae__dry__green_g: 25.1590 - mae_all: 14.1804 - val_loss: 50.5224 - val_mae__dry__clover_g: 5.4488 - val_mae__dry__dead_g: 12.6832 - val_mae__dry__green_g: 23.8651 - val_mae_all: 13.9990 - learning_rate: 1.0000e-04
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 744ms/step - loss: 51.8619 - mae__dry__clover_g: 6.4140 - mae__dry__dead_g: 10.5679 - mae__dry__green_g: 23.6957 - mae_all: 13.5592 - val_loss: 46.2020 - val_mae__dry__clover_g: 5.4536 - val_mae__dry__dead_g: 12.5327 - val_mae__dry__green_g: 22.1795 - val_mae_all: 13.3886 - learning_rate: 1.0000e-04
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 746ms/step - loss: 47.8612 - mae__dry__clover_g: 6.5708 - mae__dry__dead_g: 10.2293 - mae__dry__green_g: 22.0893 - mae_all: 12.9632 - val_loss: 42.1824 - val_mae__dry__clover_g: 5.4703 - val_mae__dry__dead_g: 12.1251 - val_mae__dry__green_g: 20.5345 - val_mae_all: 12.7100 - learning

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 31s 785ms/step - loss: 58.7631 - mae__dry__clover_g: 6.4325 - mae__dry__dead_g: 11.7326 - mae__dry__green_g: 25.5030 - mae_all: 14.5560 - val_loss: 54.5899 - val_mae__dry__clover_g: 7.1105 - val_mae__dry__dead_g: 11.1944 - val_mae__dry__green_g: 24.5147 - val_mae_all: 14.2732 - learning_rate: 1.0000e-04
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 783ms/step - loss: 54.2715 - mae__dry__clover_g: 6.4106 - mae__dry__dead_g: 11.4178 - mae__dry__green_g: 23.6236 - mae_all: 13.8173 - val_loss: 51.7347 - val_mae__dry__clover_g: 7.1050 - val_mae__dry__dead_g: 10.8107 - val_mae__dry__green_g: 23.4681 - val_mae_all: 13.7946 - learning_rate: 1.0000e-04
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 781ms/step - loss: 49.4269 - mae__dry__clover_g: 6.4181 - mae__dry__dead_g: 10.8817 - mae__dry__green_g: 21.6685 - mae_all: 12.9894 - val_loss: 48.4245 - val_mae__dry__clover_g: 7.0897 - val_mae__dry__dead_g: 10.1509 - val_mae__dry__green_g: 22.3198 - val_mae_all: 13.1868 - learning

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 748ms/step - loss: 54.0711 - mae__dry__clover_g: 6.2655 - mae__dry__dead_g: 11.4536 - mae__dry__green_g: 24.5336 - mae_all: 14.0842 - val_loss: 60.5783 - val_mae__dry__clover_g: 7.5129 - val_mae__dry__dead_g: 8.7277 - val_mae__dry__green_g: 25.9225 - val_mae_all: 14.0544 - learning_rate: 1.0000e-04
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 49.0114 - mae__dry__clover_g: 6.2181 - mae__dry__dead_g: 10.7738 - mae__dry__green_g: 22.7322 - mae_all: 13.2414 - val_loss: 49.6305 - val_mae__dry__clover_g: 7.4465 - val_mae__dry__dead_g: 7.9306 - val_mae__dry__green_g: 22.2184 - val_mae_all: 12.5318 - learning_rate: 1.0000e-04
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 43.3063 - mae__dry__clover_g: 6.2212 - mae__dry__dead_g: 10.0166 - mae__dry__green_g: 20.6670 - mae_all: 12.3016 - val_loss: 40.5724 - val_mae__dry__clover_g: 7.3607 - val_mae__dry__dead_g: 7.7535 - val_mae__dry__green_g: 19.0472 - val_mae_all: 11.3871 - learning_ra

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 753ms/step - loss: 58.0577 - mae__dry__clover_g: 6.3529 - mae__dry__dead_g: 10.7210 - mae__dry__green_g: 25.5077 - mae_all: 14.1939 - val_loss: 47.6903 - val_mae__dry__clover_g: 7.1609 - val_mae__dry__dead_g: 11.9448 - val_mae__dry__green_g: 21.9078 - val_mae_all: 13.6711 - learning_rate: 1.0000e-04
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 746ms/step - loss: 52.9847 - mae__dry__clover_g: 6.3462 - mae__dry__dead_g: 10.2933 - mae__dry__green_g: 23.7845 - mae_all: 13.4747 - val_loss: 39.7969 - val_mae__dry__clover_g: 7.1100 - val_mae__dry__dead_g: 11.0316 - val_mae__dry__green_g: 18.8440 - val_mae_all: 12.3285 - learning_rate: 1.0000e-04
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 747ms/step - loss: 47.5655 - mae__dry__clover_g: 6.3560 - mae__dry__dead_g: 9.7347 - mae__dry__green_g: 21.8751 - mae_all: 12.6553 - val_loss: 33.0652 - val_mae__dry__clover_g: 7.0279 - val_mae__dry__dead_g: 10.3567 - val_mae__dry__green_g: 16.3260 - val_mae_all: 11.2369 - learning_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 747ms/step - loss: 57.3953 - mae__dry__clover_g: 6.7009 - mae__dry__dead_g: 11.5420 - mae__dry__green_g: 25.3917 - mae_all: 14.5449 - val_loss: 60.8814 - val_mae__dry__clover_g: 5.2843 - val_mae__dry__dead_g: 11.0424 - val_mae__dry__green_g: 26.3610 - val_mae_all: 14.2292 - learning_rate: 1.0000e-04
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 744ms/step - loss: 52.6194 - mae__dry__clover_g: 6.6693 - mae__dry__dead_g: 11.0581 - mae__dry__green_g: 23.6913 - mae_all: 13.8063 - val_loss: 54.5248 - val_mae__dry__clover_g: 5.2370 - val_mae__dry__dead_g: 10.5508 - val_mae__dry__green_g: 23.9524 - val_mae_all: 13.2467 - learning_rate: 1.0000e-04
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 46.6340 - mae__dry__clover_g: 6.6192 - mae__dry__dead_g: 10.3592 - mae__dry__green_g: 21.5387 - mae_all: 12.8390 - val_loss: 46.3278 - val_mae__dry__clover_g: 5.1773 - val_mae__dry__dead_g: 9.4457 - val_mae__dry__green_g: 20.6296 - val_mae_all: 11.7508 - learning_

--------------------------------------

In [ ]:
def train_one_fold_student_distill(
    fold,
    teacher_model,
    alpha=0.4,
    tau=3.0,
    target_list=None,
    model_fn=biomass_student_from_baseline,
    **model_kwargs,
):
    """Train a distilled student for one fold using a fixed teacher model.

    Args:
        fold: Fold index used as validation split.
        teacher_model: Trained teacher model (image + metadata) used to
            generate soft labels.
        alpha: Distillation weight (teacher term) passed to the loss.
        tau: Distillation temperature passed to the loss.
        target_list: List of target column names; if None, defaults to TRAIN_TARGETS.
        model_fn: Function that builds the student model.
        **model_kwargs: Extra keyword arguments forwarded to `model_fn`.

    Returns:
        tuple: (best_val_mae, student_model, history, val_ds, w_r2)
    """
    
    if target_list is None:
        target_list = TRAIN_TARGETS
    n_targets = len(target_list)

    print(f"\n===== [Student Distill] Fold {fold} (alpha={alpha}, tau={tau}) =====")

    # 1) Build data module and collect all data
    dm = BiomassImgDataModule(fold=fold, target_list=target_list, include_meta=True)
    imgs_all = dm.make_arrays()          # (N, H, W, 3)
    y_gt_all = dm.y                     # (N, T)

    meta_cont_all = dm.df[dm.meta_cols_cont].astype("float32").values
    meta_cat_all  = dm.df[dm.meta_cols_cat ].astype("int32").values

    # 2) Teacher predictions for all samples 
    x_all = {
        "image":     imgs_all,
        "meta_cont": meta_cont_all,
        "meta_cat":  meta_cat_all,
    }
    teacher_preds_all = teacher_model.predict(
        x_all,
        batch_size=CFG.batch_size,
        verbose=1,   # (N, T)
    )                                 

    # 3) Concatenate GT and teacher predictions: y_true_concat = [y_gt, y_teacher]
    y_concat_all = np.concatenate([y_gt_all, teacher_preds_all], axis=1)  # (N, 2T)

    # 4) Train / Val split
    val_mask = (dm.df["fold"] == fold).values
    tr_idx = np.where(~val_mask)[0]
    va_idx = np.where(val_mask)[0]

    print(f"[Student] Fold {fold} Train: {len(tr_idx)} | Val: {len(va_idx)}")

    train_ds = tf.data.Dataset.from_tensor_slices(
        (imgs_all[tr_idx], y_concat_all[tr_idx])
    )
    val_ds = tf.data.Dataset.from_tensor_slices(
        (imgs_all[va_idx], y_concat_all[va_idx])
    )

    train_ds = (
        train_ds
        .map(tfdata_augmentor, num_parallel_calls=1)
        .shuffle(1024, seed=CFG.seed, reshuffle_each_iteration=False)
        .batch(CFG.batch_size)
        .prefetch(1)
    )
    val_ds = (
        val_ds
        .map(tfdata_clip, num_parallel_calls=1)
        .batch(CFG.batch_size)
        .prefetch(1)
    )

    # 5) Build student-from-baseline + distillation loss
    mode = "log1p" if CFG.use_log1p else "gram"
    baseline_ckpt = f"./ckpt/baseline_{mode}_fold{fold}.weights.h5"

    student_model = model_fn(
        image_size_h=CFG.image_size_h,
        image_size_w=CFG.image_size_w,
        target_names=target_list,
        baseline_weights_path=baseline_ckpt,
        distill_alpha=alpha, # teacher weight in distillation loss
        distill_tau=tau,      # biomass_student_from_baseline 
        use_distill_loss=True,
        **model_kwargs,
    )

    # 6) callbacks
    callbacks_list = build_callbacks(monitor="val_mae_all", mode="min")

    # 7) train
    history = student_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        callbacks=callbacks_list,
        verbose=1,
    )

    best_val_mae = min(history.history["val_mae_all"])
    print(f"[Student] Fold {fold} best val_mae_all = {best_val_mae:.3f}")

    # 8) Evaluate in gram space
    val_df, x_val = build_val_inputs(
        dm=dm,
        fold=fold,
        include_meta=False,   # student uses image only for inference
    )
    y_pred_default = student_model.predict(x_val, batch_size=CFG.batch_size)

    w_r2, pred_df, true_df = eval_weighted_r2_in_gram(
        val_df=val_df,
        y_pred_default=y_pred_default,
        model_name="Student",
        fold=fold,
    )

    return best_val_mae, student_model, history, val_ds, w_r2


In [ ]:
# ====== Train model: Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram = {}
student_fold_maes_gram = []
student_fold_r2_gram = []

# 🔴 give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        train_one_fold_student_distill(
            fold=f,
            teacher_model=teacher_models_gram[f],   # trained teacher for this fold
            alpha=0.2, #alpha = 0.2 → GT 0.8, Teacher 0.2
            tau=1.0, 
        )
    student_results_gram[f] = (student_mae, student_r2)
    student_fold_maes_gram.append(student_mae)
    student_fold_r2_gram.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram, 3))
print("R2  per fold:", np.round(student_fold_r2_gram, 3))
print("mean R2 =", np.mean(student_fold_r2_gram))

# Notes:
# Teacher (with meta) R² ≈ 0.70
# Image-only baseline R² ≈ 0.27
# Teacher is much stronger, but student has no metadata, so it cannot fully
# match the teacher. Using 20–30% teacher signal (alpha ≈ 0.2–0.3) is a
# reasonable “soft supervision” that helps without ignoring ground truth.

[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity

===== [Student Distill] Fold 0 (alpha=0.2, tau=1.0) =====
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 194ms/step
[Student] Fold 0 Train: 285 | Val: 72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/30


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 759ms/step - loss: 54.0498 - mae__dry__clover_g: 6.4776 - mae__dry__dead_g: 10.9020 - mae__dry__green_g: 25.1498 - mae_all: 14.1765 - val_loss: 48.5074 - val_mae__dry__clover_g: 5.4473 - val_mae__dry__dead_g: 12.6838 - val_mae__dry__green_g: 23.8393 - val_mae_all: 13.9902 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 761ms/step - loss: 49.4389 - mae__dry__clover_g: 6.4108 - mae__dry__dead_g: 10.5694 - mae__dry__green_g: 23.6612 - mae_all: 13.5472 - val_loss: 44.0813 - val_mae__dry__clover_g: 5.4526 - val_mae__dry__dead_g: 12.5448 - val_mae__dry__green_g: 22.1256 - val_mae_all: 13.3744 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 752ms/step - loss: 45.3832 - mae__dry__clover_g: 6.5761 - mae__dry__dead_g: 10.2297 - mae__dry__green_g: 22.0310 - mae_all: 12.9456 - val_loss: 40.0402 - val_mae__dry__clover_g: 5.4714 - val_mae__dry__dead_g: 12.1327 - val_mae__dry__green_g: 20.4991 - val_mae_all: 12.7010 - learning

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 756ms/step - loss: 55.4174 - mae__dry__clover_g: 6.4323 - mae__dry__dead_g: 11.7286 - mae__dry__green_g: 25.4977 - mae_all: 14.5529 - val_loss: 52.0355 - val_mae__dry__clover_g: 7.1093 - val_mae__dry__dead_g: 11.1757 - val_mae__dry__green_g: 24.5161 - val_mae_all: 14.2670 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 750ms/step - loss: 51.0070 - mae__dry__clover_g: 6.4099 - mae__dry__dead_g: 11.3899 - mae__dry__green_g: 23.6181 - mae_all: 13.8060 - val_loss: 49.1955 - val_mae__dry__clover_g: 7.1027 - val_mae__dry__dead_g: 10.7291 - val_mae__dry__green_g: 23.4748 - val_mae_all: 13.7689 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 759ms/step - loss: 46.2787 - mae__dry__clover_g: 6.4194 - mae__dry__dead_g: 10.8167 - mae__dry__green_g: 21.6826 - mae_all: 12.9729 - val_loss: 45.9140 - val_mae__dry__clover_g: 7.0845 - val_mae__dry__dead_g: 9.9913 - val_mae__dry__green_g: 22.3470 - val_mae_all: 13.1409 - learning_

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 762ms/step - loss: 51.8830 - mae__dry__clover_g: 6.2662 - mae__dry__dead_g: 11.4432 - mae__dry__green_g: 24.5280 - mae_all: 14.0791 - val_loss: 56.7995 - val_mae__dry__clover_g: 7.5115 - val_mae__dry__dead_g: 8.7101 - val_mae__dry__green_g: 25.8786 - val_mae_all: 14.0334 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 753ms/step - loss: 46.7488 - mae__dry__clover_g: 6.2195 - mae__dry__dead_g: 10.7351 - mae__dry__green_g: 22.7079 - mae_all: 13.2209 - val_loss: 45.6926 - val_mae__dry__clover_g: 7.4362 - val_mae__dry__dead_g: 7.8979 - val_mae__dry__green_g: 22.0811 - val_mae_all: 12.4717 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 755ms/step - loss: 40.9343 - mae__dry__clover_g: 6.2270 - mae__dry__dead_g: 9.9451 - mae__dry__green_g: 20.6219 - mae_all: 12.2647 - val_loss: 36.4578 - val_mae__dry__clover_g: 7.3437 - val_mae__dry__dead_g: 7.7464 - val_mae__dry__green_g: 18.7548 - val_mae_all: 11.2816 - learning_rat

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 763ms/step - loss: 54.1195 - mae__dry__clover_g: 6.3526 - mae__dry__dead_g: 10.7164 - mae__dry__green_g: 25.5061 - mae_all: 14.1917 - val_loss: 44.5679 - val_mae__dry__clover_g: 7.1605 - val_mae__dry__dead_g: 11.9547 - val_mae__dry__green_g: 21.9305 - val_mae_all: 13.6819 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 756ms/step - loss: 49.1868 - mae__dry__clover_g: 6.3455 - mae__dry__dead_g: 10.2803 - mae__dry__green_g: 23.7795 - mae_all: 13.4685 - val_loss: 36.8685 - val_mae__dry__clover_g: 7.1084 - val_mae__dry__dead_g: 11.0424 - val_mae__dry__green_g: 18.8843 - val_mae_all: 12.3450 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 753ms/step - loss: 43.9178 - mae__dry__clover_g: 6.3553 - mae__dry__dead_g: 9.7125 - mae__dry__green_g: 21.8680 - mae_all: 12.6453 - val_loss: 30.5362 - val_mae__dry__clover_g: 7.0292 - val_mae__dry__dead_g: 10.4071 - val_mae__dry__green_g: 16.4787 - val_mae_all: 11.3050 - learning_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 771ms/step - loss: 56.3040 - mae__dry__clover_g: 6.7009 - mae__dry__dead_g: 11.5331 - mae__dry__green_g: 25.3801 - mae_all: 14.5380 - val_loss: 58.5134 - val_mae__dry__clover_g: 5.2827 - val_mae__dry__dead_g: 11.0266 - val_mae__dry__green_g: 26.3272 - val_mae_all: 14.2122 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 769ms/step - loss: 51.4102 - mae__dry__clover_g: 6.6719 - mae__dry__dead_g: 11.0233 - mae__dry__green_g: 23.6480 - mae_all: 13.7811 - val_loss: 51.8651 - val_mae__dry__clover_g: 5.2362 - val_mae__dry__dead_g: 10.4920 - val_mae__dry__green_g: 23.8088 - val_mae_all: 13.1790 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 794ms/step - loss: 45.2727 - mae__dry__clover_g: 6.6249 - mae__dry__dead_g: 10.2819 - mae__dry__green_g: 21.4611 - mae_all: 12.7893 - val_loss: 43.0605 - val_mae__dry__clover_g: 5.1830 - val_mae__dry__dead_g: 9.2362 - val_mae__dry__green_g: 20.2803 - val_mae_all: 11.5665 - learning_

In [ ]:
# ====== Train model: Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram_2 = {}
student_fold_maes_gram_2 = []
student_fold_r2_gram_2 = []

# 🔴 give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        train_one_fold_student_distill(
            fold=f,
            teacher_model=teacher_models_gram[f],   # trained teacher
            alpha=0.5,
            tau=1.0,
        )
    student_results_gram_2[f] = (student_mae, student_r2)
    student_fold_maes_gram_2.append(student_mae)
    student_fold_r2_gram_2.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram_2, 3))
print("R2  per fold:", np.round(student_fold_r2_gram_2, 3))
print("mean R2 =", np.mean(student_fold_r2_gram_2))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
[init] fallback label_transform=identity

===== [Student Distill] Fold 0 (alpha=0.5, tau=1.0) =====
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 195ms/step
[Student] Fold 0 Train: 285 | Val: 72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/30


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 755ms/step - loss: 50.4779 - mae__dry__clover_g: 6.4744 - mae__dry__dead_g: 10.9047 - mae__dry__green_g: 25.1380 - mae_all: 14.1724 - val_loss: 45.5881 - val_mae__dry__clover_g: 5.4463 - val_mae__dry__dead_g: 12.7011 - val_mae__dry__green_g: 23.8389 - val_mae_all: 13.9954 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 751ms/step - loss: 45.8285 - mae__dry__clover_g: 6.4054 - mae__dry__dead_g: 10.5761 - mae__dry__green_g: 23.6198 - mae_all: 13.5338 - val_loss: 41.1840 - val_mae__dry__clover_g: 5.4525 - val_mae__dry__dead_g: 12.5803 - val_mae__dry__green_g: 22.1848 - val_mae_all: 13.4059 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 752ms/step - loss: 41.6971 - mae__dry__clover_g: 6.5847 - mae__dry__dead_g: 10.2396 - mae__dry__green_g: 21.9611 - mae_all: 12.9285 - val_loss: 37.2778 - val_mae__dry__clover_g: 5.4820 - val_mae__dry__dead_g: 12.2529 - val_mae__dry__green_g: 20.6667 - val_mae_all: 12.8005 - learning

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 761ms/step - loss: 50.4067 - mae__dry__clover_g: 6.4324 - mae__dry__dead_g: 11.7209 - mae__dry__green_g: 25.4941 - mae_all: 14.5491 - val_loss: 48.2121 - val_mae__dry__clover_g: 7.1082 - val_mae__dry__dead_g: 11.1445 - val_mae__dry__green_g: 24.5236 - val_mae_all: 14.2587 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 754ms/step - loss: 46.1299 - mae__dry__clover_g: 6.4091 - mae__dry__dead_g: 11.3484 - mae__dry__green_g: 23.6210 - mae_all: 13.7928 - val_loss: 45.3771 - val_mae__dry__clover_g: 7.0995 - val_mae__dry__dead_g: 10.5973 - val_mae__dry__green_g: 23.4894 - val_mae_all: 13.7287 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 754ms/step - loss: 41.5683 - mae__dry__clover_g: 6.4197 - mae__dry__dead_g: 10.7231 - mae__dry__green_g: 21.7272 - mae_all: 12.9567 - val_loss: 42.0732 - val_mae__dry__clover_g: 7.0780 - val_mae__dry__dead_g: 9.7366 - val_mae__dry__green_g: 22.3775 - val_mae_all: 13.0640 - learning_

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 761ms/step - loss: 48.6093 - mae__dry__clover_g: 6.2675 - mae__dry__dead_g: 11.4284 - mae__dry__green_g: 24.5236 - mae_all: 14.0732 - val_loss: 51.1824 - val_mae__dry__clover_g: 7.5086 - val_mae__dry__dead_g: 8.6891 - val_mae__dry__green_g: 25.8316 - val_mae_all: 14.0098 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 754ms/step - loss: 43.3941 - mae__dry__clover_g: 6.2227 - mae__dry__dead_g: 10.6818 - mae__dry__green_g: 22.6903 - mae_all: 13.1983 - val_loss: 39.7561 - val_mae__dry__clover_g: 7.4139 - val_mae__dry__dead_g: 7.8418 - val_mae__dry__green_g: 21.8740 - val_mae_all: 12.3766 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 755ms/step - loss: 37.4398 - mae__dry__clover_g: 6.2406 - mae__dry__dead_g: 9.8478 - mae__dry__green_g: 20.5935 - mae_all: 12.2273 - val_loss: 30.4815 - val_mae__dry__clover_g: 7.3293 - val_mae__dry__dead_g: 7.7436 - val_mae__dry__green_g: 18.3905 - val_mae_all: 11.1545 - learning_rat

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 766ms/step - loss: 48.2258 - mae__dry__clover_g: 6.3525 - mae__dry__dead_g: 10.7087 - mae__dry__green_g: 25.5093 - mae_all: 14.1902 - val_loss: 39.8969 - val_mae__dry__clover_g: 7.1601 - val_mae__dry__dead_g: 11.9671 - val_mae__dry__green_g: 21.9770 - val_mae_all: 13.7014 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 764ms/step - loss: 43.5341 - mae__dry__clover_g: 6.3455 - mae__dry__dead_g: 10.2563 - mae__dry__green_g: 23.7944 - mae_all: 13.4654 - val_loss: 32.6471 - val_mae__dry__clover_g: 7.1094 - val_mae__dry__dead_g: 11.0527 - val_mae__dry__green_g: 19.0509 - val_mae_all: 12.4043 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 758ms/step - loss: 38.5148 - mae__dry__clover_g: 6.3548 - mae__dry__dead_g: 9.6732 - mae__dry__green_g: 21.8962 - mae_all: 12.6414 - val_loss: 26.9440 - val_mae__dry__clover_g: 7.0322 - val_mae__dry__dead_g: 10.4736 - val_mae__dry__green_g: 16.8544 - val_mae_all: 11.4534 - learning_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 760ms/step - loss: 54.6821 - mae__dry__clover_g: 6.7010 - mae__dry__dead_g: 11.5243 - mae__dry__green_g: 25.3674 - mae_all: 14.5309 - val_loss: 54.9740 - val_mae__dry__clover_g: 5.2796 - val_mae__dry__dead_g: 11.0092 - val_mae__dry__green_g: 26.2813 - val_mae_all: 14.1900 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 754ms/step - loss: 49.6488 - mae__dry__clover_g: 6.6755 - mae__dry__dead_g: 10.9817 - mae__dry__green_g: 23.6024 - mae_all: 13.7532 - val_loss: 47.8144 - val_mae__dry__clover_g: 5.2358 - val_mae__dry__dead_g: 10.4180 - val_mae__dry__green_g: 23.5668 - val_mae_all: 13.0735 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 754ms/step - loss: 43.3043 - mae__dry__clover_g: 6.6354 - mae__dry__dead_g: 10.1894 - mae__dry__green_g: 21.3729 - mae_all: 12.7326 - val_loss: 38.1910 - val_mae__dry__clover_g: 5.1957 - val_mae__dry__dead_g: 8.9637 - val_mae__dry__green_g: 19.7683 - val_mae_all: 11.3092 - learning_

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# Define data for the conclusion
models = [
    'Mean Baseline,',
    'CNN Baseline, 15 epochs, ~33mins',
    'Teacher Model, 15 epchs, ~37mins',
    'Student (Fine-tuned only), 15 epochs, ~35mins',
    'Distilled Student (α=0.2, τ=1.0), 30 epochs, ~60mins',
    'Distilled Student (α=0.5, τ=1.0), 30 epochs, ~62mins',   
]

inputs = [
    'N/A',
    'Image Only',
    'Image + Metadata',
    'Image Only',
    'Image Only',
    'Image Only',
]

# Updated R² values from your notebook results
r2_scores = [
    0.243,
    0.270,
    0.699,
    0.372,
    0.432,
    0.440,
]

insights = [
    'The absolute minimum benchmark (predicts using training set means only).',
    'Standard deep learning approach (trained from scratch) yields only modest gains.',
    'The performance ceiling. Confirms that metadata contains critical signals for this task.',
    'Fine-tuning from baseline weights helps, but hits a low ceiling without distillation.',
    'Distillation with α=0.2 provides a significant boost to the image-only model.',
    'Best image-only result. Higher teacher weight (α=0.5) successfully transfers "dark knowledge."',
]

# ============== Print Summary Statistics ==============
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)

for i, model in enumerate(models):
    print(f"\n{model}")
    print(f"  Inputs: {inputs[i]}")
    print(f"  Mean Weighted R²: {r2_scores[i]:.3f}")
    print(f"  Insight: {insights[i]}")

print("\n" + "="*60)
print("KEY FINDINGS:")
print("="*60)
print(f"• Improvement from Mean Baseline → Best Distilled Student: "
      f"{((r2_scores[5] - r2_scores[0]) / r2_scores[0] * 100):.1f}%")
print(f"• Performance gap (Best Student vs Teacher): "
      f"{(r2_scores[2] - r2_scores[5]):.3f}")
print(f"• Best α=0.5 outperforms α=0.2 by: "
      f"{((r2_scores[5] - r2_scores[4]) / r2_scores[4] * 100):.1f}%")

print("\n" + "="*60)
print("Main challenges:")
print("="*60)
print(f"• Limited dataset size (357 images) ")
print(f"• image only as input model")

print("\n" + "="*60)
print("Future work:")
print("="*60)
print(f"• Explore different alpha and tau schedules")
print(f"• test more augmentation")
print(f"• Use stronger image backbones (ResNet / EfficientNet with pretraining) and then distill again.")


MODEL PERFORMANCE SUMMARY

Mean Baseline,
  Inputs: N/A
  Mean Weighted R²: 0.243
  Insight: The absolute minimum benchmark (predicts using training set means only).

CNN Baseline, 15 epochs, ~33mins
  Inputs: Image Only
  Mean Weighted R²: 0.270
  Insight: Standard deep learning approach (trained from scratch) yields only modest gains.

Teacher Model, 15 epchs, ~37mins
  Inputs: Image + Metadata
  Mean Weighted R²: 0.699
  Insight: The performance ceiling. Confirms that metadata contains critical signals for this task.

Student (Fine-tuned only), 15 epochs, ~35mins
  Inputs: Image Only
  Mean Weighted R²: 0.372
  Insight: Fine-tuning from baseline weights helps, but hits a low ceiling without distillation.

Distilled Student (α=0.2, τ=1.0), 30 epochs, ~60mins
  Inputs: Image Only
  Mean Weighted R²: 0.432
  Insight: Distillation with α=0.2 provides a significant boost to the image-only model.

Distilled Student (α=0.5, τ=1.0), 30 epochs, ~62mins
  Inputs: Image Only
  Mean Weighted R